In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [23]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score



In [24]:
df=pd.read_csv('Titanic.csv')[['Fare','Age','Survived']]

In [25]:
df.dropna(inplace = True)
x= df.drop(columns=['Survived'])
y=df['Survived']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

### Kmeans

In [60]:
kbins= KBinsDiscretizer(n_bins = 4, encode='ordinal', strategy='kmeans')

In [67]:
trf = ColumnTransformer([
    ('kmeans', kbins, ['Fare','Age'])
],remainder= 'passthrough')

In [68]:
x_train_transform1 = trf.fit_transform(x_train)
x_test_transform1 = trf.transform(x_test)


#### Before Kmeans

In [69]:
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)
y_pred = clf.predict(x_test)
accuracy_score(y_test,y_pred)


0.4888888888888889

In [70]:
np.mean(cross_val_score(clf,x,y,cv=10, scoring='accuracy'))

np.float64(0.5122222222222221)

### After Kmeans

In [71]:
clf = DecisionTreeClassifier()
clf.fit(x_train_transform1, y_train)
y_pred=clf.predict(x_test_transform1)
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.6277777777777778

In [72]:
np.mean(cross_val_score(clf,x_train_transform1,y_train,cv=10,scoring='accuracy'))

np.float64(0.6305555555555555)

### As a result we improve our model by almost 12%

### Now quantile

In [119]:
kbins_quantile= KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile' , quantile_method='averaged_inverted_cdf')
trf2 = ColumnTransformer([
    ('quantile',kbins_quantile,['Fare','Age'])
],remainder = 'passthrough')

In [120]:
x_train_quantile = trf2.fit_transform(x_train)
x_test_quantile = trf2.transform(x_test)
x_train_quantile = pd.DataFrame(x_train_quantile, columns=x_train.columns )
x_test_quantile = pd.DataFrame(x_test_quantile, columns=x_test.columns)


In [121]:
clf2 = DecisionTreeClassifier()
clf2.fit(x_train_quantile,y_train)
y_pred=clf2.predict(x_test_quantile)
accuracy_score(y_test,y_pred)


0.6222222222222222

In [122]:
np.mean(cross_val_score(clf2,x_train_quantile,y_train,cv=10, scoring='accuracy'))

np.float64(0.5999999999999999)

### Performance is increased than without discretization but not good as Kmeans

### Now last one Uniform

In [136]:
kbins_uniform = KBinsDiscretizer(n_bins=5, encode= 'ordinal', strategy='uniform')


In [137]:
trf3 = ColumnTransformer([
    ('uniform',kbins_uniform,['Fare','Age'])
],remainder = 'passthrough')

In [138]:
x_train_uniform = trf3.fit_transform(x_train)
x_test_uniform = trf3.transform(x_test)
x_train_uniform = pd.DataFrame(x_train_uniform, columns=x_train.columns)
x_test_uniform = pd.DataFrame(x_test_uniform,columns=x_test.columns)

In [139]:
clf3 = DecisionTreeClassifier()
clf3.fit(x_train_uniform,y_train)
y_pred=clf3.predict(x_test_uniform)
accuracy_score(y_pred,y_test)

0.6388888888888888

In [140]:
np.mean(cross_val_score(clf3,x_train_uniform,y_train,cv=10,scoring ='accuracy'))

np.float64(0.6222222222222222)

### So, For this 